In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)
import joblib

print("All imports loaded. XGBoost version:", xgb.__version__)

All imports loaded. XGBoost version: 3.2.0


In [2]:
# German Credit dataset — 1,000 loan applicants labeled good/bad credit risk
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"

columns = [
    "checking_account", "duration_months", "credit_history", "purpose",
    "credit_amount", "savings_account", "employment_since", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since", "property",
    "age", "other_installment_plans", "housing", "existing_credits",
    "job", "num_dependents", "telephone", "foreign_worker", "target"
]

df = pd.read_csv(url, sep=" ", header=None, names=columns)
print("Shape:", df.shape)
df.head()

Shape: (1000, 21)


,checking_account,duration_months,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [3]:
# Know your data — the step that separates engineers from button-pushers
print("Class balance (1=good, 2=bad):")
print(df["target"].value_counts(), "\n")

print("Missing values:", df.isnull().sum().sum())
print("\nNumeric columns summary:")
df.describe()

Class balance (1=good, 2=bad):
target
1    700
2    300
Name: count, dtype: int64 

Missing values: 0

Numeric columns summary:


,duration_months,credit_amount,installment_rate,residence_since,age,existing_credits,num_dependents,target
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000,1.300000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086,0.458487
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000,1.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000,1.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000,2.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000,2.000000


In [4]:
# Separate features from target
X = df.drop("target", axis=1)
y = df["target"].map({1: 0, 2: 1})  # remap: 1=default/bad risk (the positive class), 0=good

# Encode categorical columns (the A11, A34 codes) into numbers
categorical_cols = X.select_dtypes(include="object").columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("Original feature count:", X.shape[1])
print("After encoding:", X_encoded.shape[1])

# Train/test split — stratified to preserve the 70/30 class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTrain size:", X_train.shape[0], "| Test size:", X_test.shape[0])
print("Default rate in train:", round(y_train.mean(), 3), "| test:", round(y_test.mean(), 3))

Original feature count: 20
After encoding: 48

Train size: 800 | Test size: 200
Default rate in train: 0.3 | test: 0.3


/var/folders/nw/j4d0ymn52vx4nztfyw97cwrh0000gn/T/ipykernel_33981/4259225441.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns


In [5]:
# Handle class imbalance: weight the minority class (defaulters) up
# ratio of negatives to positives tells XGBoost to take defaults more seriously
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(scale_pos_weight, 2))

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained.")

scale_pos_weight: 2.33
Model trained.


In [6]:
# Predictions
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # probability of default

# Metrics that matter for credit risk
print("Accuracy: ", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3), "(of those flagged as default, how many really were)")
print("Recall:   ", round(recall_score(y_test, y_pred), 3), "(of actual defaulters, how many we caught)")
print("AUC:      ", round(roc_auc_score(y_test, y_proba), 3), "(overall ranking quality)")

print("\nConfusion matrix:")
print(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual_good", "actual_default"],
    columns=["pred_good", "pred_default"]
))

print("\nFull report:")
print(classification_report(y_test, y_pred, target_names=["good", "default"]))

Accuracy:  0.71
Precision: 0.512 (of those flagged as default, how many really were)
Recall:    0.7 (of actual defaulters, how many we caught)
AUC:       0.793 (overall ranking quality)

Confusion matrix:
                pred_good  pred_default
actual_good           100            40
actual_default         18            42

Full report:
              precision    recall  f1-score   support

        good       0.85      0.71      0.78       140
     default       0.51      0.70      0.59        60

    accuracy                           0.71       200
   macro avg       0.68      0.71      0.68       200
weighted avg       0.75      0.71      0.72       200



In [7]:
import os
os.makedirs("../models", exist_ok=True)

# Save the model AND the feature columns (critical — scoring needs the exact same columns)
joblib.dump(model, "../models/credit_risk_model.pkl")
joblib.dump(list(X_encoded.columns), "../models/feature_columns.pkl")

print("Saved model and feature columns to ../models/")
print("Feature count saved:", len(X_encoded.columns))

Saved model and feature columns to ../models/
Feature count saved: 48


In [8]:
import os
print("Notebook is in:", os.getcwd())
print("Models folder exists:", os.path.exists("../models"))
print("Files:", os.listdir("../models") if os.path.exists("../models") else "NOT FOUND")

Notebook is in: /Users/collinsagyekum/Documents/credit-risk-agentic-ai
Models folder exists: True
Files: ['feature_columns.pkl', 'credit_risk_model.pkl']


In [9]:
import os
print("Notebook is in:", os.getcwd())
print("Files in ../models:", os.listdir("../models") if os.path.exists("../models") else "NOT FOUND")

Notebook is in: /Users/collinsagyekum/Documents/credit-risk-agentic-ai
Files in ../models: ['feature_columns.pkl', 'credit_risk_model.pkl']
